In [1]:
import torch, torchvision
import torch.nn as nn, torch.optim as optim
import torchvision.transforms as T 
from torch.utils.data import DataLoader, Subset

In [2]:
t = T.Compose([
    T.Resize((224,224)),
    T.Grayscale(3),
    T.ToTensor(),
    T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trd = DataLoader(Subset(torchvision.datasets.MNIST('./data',True,transform=t,download=True),range(2000)),64,True)
ted = DataLoader(Subset(torchvision.datasets.MNIST('./data',False,transform=t,download=True),range(500)),64)

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw



In [3]:
class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.feats = nn.Sequential(
            nn.Conv2d(3,64,11,4,2), nn.ReLU(), nn.MaxPool2d(3,2),
            nn.Conv2d(64,192,5,padding=2), nn.ReLU(), nn.MaxPool2d(3,2),
            nn.Conv2d(192,384,3,padding=1), nn.ReLU(),
            nn.Conv2d(384,256,3,padding=1), nn.ReLU(),
            nn.Conv2d(256,256,3,padding=1), nn.ReLU(), nn.MaxPool2d(3,2)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(), nn.Linear(9216,4096),
            nn.Dropout(), nn.Linear(4096,4096), nn.ReLU(),
            nn.Linear(4096,10)
        )
    def forward(self,x):
        x = self.feats(x)
        x = x.view(x.size(0),-1)
        x = self.classifier(x)
        return x

In [4]:
model = AlexNet()
opt = optim.Adam(model.parameters(),lr=0.001)
crit = nn.CrossEntropyLoss()

for e in range(1):
    model.train()
    rl=0
    for i,(x,y) in enumerate(trd):
        opt.zero_grad()
        o = model(x)
        l = crit(o,y)
        l.backward()
        opt.step()
        rl += l.item()
        if i%100 == 0:
            print(f'batch {i+1}: Loss: {rl:.2f}')
            rl=0

batch 1: Loss: 2.30


In [7]:
model.eval()
c=n=0
with torch.no_grad():
    for x,y in ted:

        c = (model(x).argmax(1)==y).sum().item()
        n = len(y)
print(f'Accuracy: {c/n*100:.2f}%')

Accuracy: 5.77%
